# Aligned vs Non-Aligned Ensembles

Ensemble members do not always share the same time grid. In plasma physics UQ workflows, two common scenarios arise:

1. **Aligned ensemble** — all members start at $t = 0$ and use the same time grid. Trimming to a shared start time ensures every member contributes data from the same physical phase of the simulation.

2. **Non-aligned ensemble** — members start at different times, run to different end times, or have different sampling rates. Each member is trimmed independently based on its own convergence behavior.

`quends` handles both cases. This notebook demonstrates each approach and shows how to compare the resulting UQ estimates.

**Which approach should I use?**

| Scenario | Recommended approach |
|----------|---------------------|
| All members share the same time grid | Aligned — enforces a common start time |
| Members have different lengths or start times | Non-aligned — each member trimmed independently |
| You want maximum reproducibility | Aligned — results are deterministic given a fixed start time |
| You want adaptive, data-driven trimming | Non-aligned — each member uses its own convergence point |

## 1. Load Ensemble Members

We start by loading three GX simulation runs. Even though these members were launched with the same physical parameters, they may differ in length (e.g., if one run was restarted or ran longer).

In [ ]:
import quends as qnds

members = [
    qnds.from_csv("gx/ensemble/tprim_2_5_a.out.csv"),
    qnds.from_csv("gx/ensemble/tprim_2_5_b.out.csv"),
    qnds.from_csv("gx/ensemble/tprim_2_5_c.out.csv"),
]

print("Member lengths and time ranges:")
for i, m in enumerate(members):
    print(f"  Member {i}: {len(m.data)} time steps, t_max={m.data['time'].max():.1f}")

## 2. Non-Aligned Ensemble

In the non-aligned approach, each member is trimmed **independently** using `Ensemble.trim()`. The trimming algorithm inspects each member's own time series and finds the point at which that member's signal has settled into stationarity. Members with a longer transient will be trimmed more aggressively than members that converge quickly.

This is the **default and most common** workflow in `quends`.

In [ ]:
ens = qnds.Ensemble(members)
ens_nonaligned = ens.trim("HeatFlux_st", method="std", window_size=20)

print("Non-aligned trim — each member trimmed independently:")
for i, ds in enumerate(ens_nonaligned.data_streams):
    t_start = ds.data['time'].min()
    print(f"  Member {i}: {len(ds.data)} steps remaining, trim start t={t_start:.1f}")

## 3. Aligned Ensemble

In the aligned approach, all members are trimmed to the **same start time** regardless of when each individual member would naturally converge. This is done by passing `start_time` to `QuantileTrimStrategy` directly.

You must use `TrimDataStreamOperation` directly on each member (rather than `Ensemble.trim()`) so that you can pass the shared `start_time` parameter. Members whose data does not reach `start_time` (or which are empty after trimming) are filtered out.

**Choose `start_time` carefully** — it should be after the transient for all members. A common approach is to inspect the non-aligned trim results above and pick a `start_time` that is at or after the latest trim point observed.

In [ ]:
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation

# Use a shared start_time so all members contribute data from the same point onward.
# Adjust this value to match your data; it should exceed the transient for every member.
strategy = QuantileTrimStrategy(window_size=20, robust=True, start_time=100.0)
op = TrimDataStreamOperation(strategy=strategy)

aligned_trimmed = [op(m, column_name="HeatFlux_st") for m in members]

# Filter out any members that ended up empty after applying start_time
aligned_trimmed = [t for t in aligned_trimmed if not t.data.empty]

ens_aligned = qnds.Ensemble(aligned_trimmed)

print(f"Aligned ensemble: {len(ens_aligned.data_streams)} members retained after filtering")
for i, ds in enumerate(ens_aligned.data_streams):
    t_start = ds.data['time'].min()
    print(f"  Member {i}: {len(ds.data)} steps, all starting at t>={t_start:.1f}")

## 4. Compare Aligned vs Non-Aligned Statistics

We now compute Technique 2 (member-wise IVW) statistics for both the aligned and non-aligned ensembles. If the chosen `start_time` is well within the stationary region, the two estimates should be broadly consistent — any difference reflects the different data windows used.

In [ ]:
r_aligned    = ens_aligned.compute_statistics("HeatFlux_st", technique=2)
r_nonaligned = ens_nonaligned.compute_statistics("HeatFlux_st", technique=2)

print("Technique 2 (IVW) — Aligned vs Non-Aligned:")
print("-" * 50)
print(f"  Aligned    mean : {r_aligned['mean']:.4f}")
print(f"  Aligned    CI   : {r_aligned['confidence_interval']}")
print(f"  Nonaligned mean : {r_nonaligned['mean']:.4f}")
print(f"  Nonaligned CI   : {r_nonaligned['confidence_interval']}")

## 5. Full Technique Comparison for Both Ensembles

For completeness, we compute all three techniques for both ensemble types. This provides a comprehensive picture of how the choice of alignment and analysis technique together influence the final UQ result.

In [ ]:
technique_names = {0: "Average-then-analyze", 1: "Pooled-block", 2: "Member-wise IVW"}

print(f"{'':>14}  {'T0 mean':>10}  {'T1 mean':>10}  {'T2 mean':>10}")
print("-" * 52)

for label, ensemble in [("Aligned", ens_aligned), ("Non-aligned", ens_nonaligned)]:
    means = []
    for t in [0, 1, 2]:
        r = ensemble.compute_statistics("HeatFlux_st", technique=t)
        means.append(r['mean'])
    print(f"{label:>14}  {means[0]:>10.4f}  {means[1]:>10.4f}  {means[2]:>10.4f}")

## 6. When to Use Each Approach

### Use the Aligned Approach When:

- You have **a priori knowledge** of the transient length (e.g., from a reference simulation or convergence study).
- You need **reproducible results** that do not depend on the data-driven trim algorithm.
- You are comparing across different parameter sets and want a consistent time window.
- All members share the same time grid and you want to ensure no member is trimmed more than the others.

### Use the Non-Aligned Approach When:

- Members have **different convergence rates** and you want each member trimmed adaptively.
- You do not know the transient length in advance.
- Some members may be restarted runs with a different effective start time.
- You want `quends` to do the heavy lifting of finding each member's stationary window automatically.

### Practical Recommendation

For exploratory analysis, start with the **non-aligned** approach using `Ensemble.trim()`. Once you have identified a suitable `start_time` from the per-member trim results, switch to the **aligned** approach for your final published results to maximize reproducibility.

## Summary

- **Non-aligned ensembles** use `Ensemble.trim(col, method, window_size)` — each member is trimmed adaptively. This is the most common starting point.
- **Aligned ensembles** use `QuantileTrimStrategy(window_size, robust=True, start_time=...)` + `TrimDataStreamOperation` applied member-by-member, with empty results filtered out before wrapping in `qnds.Ensemble()`.
- Both ensemble types support all three compute techniques (`technique=0, 1, 2`).
- The aligned approach gives more interpretable and reproducible results when the shared start time is well-chosen.